# EMS Data Profiling

Works out what the EPCR (Elite) data can answer before any analysis starts.

Read only. Nothing is written to any table or view. The one file produced is an Excel workbook at the end.

Every query in section 3 is a complete, standalone SQL statement. Copy any one of them straight into the SQL editor or a Genie cell and it runs as-is. Nothing is assembled from variables, so what you read is exactly what executes.

## 1. Can we read the data

In [ ]:
try:
    n = spark.sql("SELECT count(*) AS rows FROM prod.silver_elite_dwgmr.fact_incident").collect()[0]["rows"]
    print("fact_incident rows:", n)
except Exception as e:
    print(e)

The simplest possible query against the fact table, with the full error printed rather than swallowed.

The last run failed on all twenty queries, which almost always means one problem at the root rather than twenty. Run this first. If it comes back with insufficient privileges, everything below will fail the same way and it is a grant request, not a code fix.

Worth knowing: the earlier version of this notebook did read 13.3 million rows from this exact table. So either that run used a different cluster or SQL warehouse, or a grant changed since. Check which compute the working run used before assuming the access is gone.

In [ ]:
for t in ["fact_incident", "dim_incident", "dim_situation", "dim_disposition",
          "dim_agency", "dim_patient", "dim_payment", "dim_scene"]:
    try:
        print(t, len(spark.table(f"prod.silver_elite_dwgmr.{t}").columns), "columns")
    except Exception as e:
        print(t, str(e)[:150])

Checks each table the join needs, one at a time. If some read and others do not, the grant is partial and the specific tables to ask for are named here.

## 2. Column names

In [ ]:
for t in ["fact_incident", "dim_incident", "dim_situation", "dim_disposition",
          "dim_agency", "dim_patient", "dim_payment", "dim_scene"]:
    try:
        print(t)
        print([f.name for f in spark.table(f"prod.silver_elite_dwgmr.{t}").schema.fields])
        print()
    except Exception as e:
        print(t, str(e)[:150], "\n")

The columns of every table the join touches.

The SQL below names columns explicitly, so if a name differs here that is where you fix it. This is also where to find the secondary impression, the comorbidity fields, and the dispatch complaint column that would separate 911 calls from interfacility transfers.

## 3. Run the SQL

In [ ]:
QUERIES = {

"row_check": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT count(*) AS incidents, count(DISTINCT incident_id) AS distinct_incidents FROM scope
""",

"volume_by_month": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT yr, mo, count(*) AS incidents FROM scope GROUP BY yr, mo ORDER BY mo
""",

"volume_by_agency": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT yr, agency_name, count(*) AS incidents FROM scope GROUP BY yr, agency_name ORDER BY yr, incidents DESC
""",

"volume_by_state": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT state, count(*) AS incidents FROM scope GROUP BY state ORDER BY incidents DESC
""",

"completeness": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT 'incident_id' AS field,
       count(CASE WHEN incident_id IS NOT NULL AND lower(trim(cast(incident_id AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN incident_id IS NOT NULL AND lower(trim(cast(incident_id AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM scope
UNION ALL
SELECT 'incident_date' AS field,
       count(CASE WHEN incident_date IS NOT NULL AND lower(trim(cast(incident_date AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN incident_date IS NOT NULL AND lower(trim(cast(incident_date AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM scope
UNION ALL
SELECT 'patient_id' AS field,
       count(CASE WHEN patient_id IS NOT NULL AND lower(trim(cast(patient_id AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN patient_id IS NOT NULL AND lower(trim(cast(patient_id AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM scope
UNION ALL
SELECT 'payer' AS field,
       count(CASE WHEN payer IS NOT NULL AND lower(trim(cast(payer AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN payer IS NOT NULL AND lower(trim(cast(payer AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM scope
UNION ALL
SELECT 'primary_impression' AS field,
       count(CASE WHEN primary_impression IS NOT NULL AND lower(trim(cast(primary_impression AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN primary_impression IS NOT NULL AND lower(trim(cast(primary_impression AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM scope
UNION ALL
SELECT 'acuity' AS field,
       count(CASE WHEN acuity IS NOT NULL AND lower(trim(cast(acuity AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN acuity IS NOT NULL AND lower(trim(cast(acuity AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM scope
UNION ALL
SELECT 'disposition' AS field,
       count(CASE WHEN disposition IS NOT NULL AND lower(trim(cast(disposition AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN disposition IS NOT NULL AND lower(trim(cast(disposition AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM scope
UNION ALL
SELECT 'state' AS field,
       count(CASE WHEN state IS NOT NULL AND lower(trim(cast(state AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN state IS NOT NULL AND lower(trim(cast(state AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM scope
UNION ALL
SELECT 'county' AS field,
       count(CASE WHEN county IS NOT NULL AND lower(trim(cast(county AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN county IS NOT NULL AND lower(trim(cast(county AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM scope
UNION ALL
SELECT 'zip' AS field,
       count(CASE WHEN zip IS NOT NULL AND lower(trim(cast(zip AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN zip IS NOT NULL AND lower(trim(cast(zip AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM scope
UNION ALL
SELECT 'agency_name' AS field,
       count(CASE WHEN agency_name IS NOT NULL AND lower(trim(cast(agency_name AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN agency_name IS NOT NULL AND lower(trim(cast(agency_name AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM scope
ORDER BY pct
""",

"completeness_medicaid": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT 'incident_id' AS field,
       count(CASE WHEN incident_id IS NOT NULL AND lower(trim(cast(incident_id AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN incident_id IS NOT NULL AND lower(trim(cast(incident_id AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM med
UNION ALL
SELECT 'incident_date' AS field,
       count(CASE WHEN incident_date IS NOT NULL AND lower(trim(cast(incident_date AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN incident_date IS NOT NULL AND lower(trim(cast(incident_date AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM med
UNION ALL
SELECT 'patient_id' AS field,
       count(CASE WHEN patient_id IS NOT NULL AND lower(trim(cast(patient_id AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN patient_id IS NOT NULL AND lower(trim(cast(patient_id AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM med
UNION ALL
SELECT 'payer' AS field,
       count(CASE WHEN payer IS NOT NULL AND lower(trim(cast(payer AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN payer IS NOT NULL AND lower(trim(cast(payer AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM med
UNION ALL
SELECT 'primary_impression' AS field,
       count(CASE WHEN primary_impression IS NOT NULL AND lower(trim(cast(primary_impression AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN primary_impression IS NOT NULL AND lower(trim(cast(primary_impression AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM med
UNION ALL
SELECT 'acuity' AS field,
       count(CASE WHEN acuity IS NOT NULL AND lower(trim(cast(acuity AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN acuity IS NOT NULL AND lower(trim(cast(acuity AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM med
UNION ALL
SELECT 'disposition' AS field,
       count(CASE WHEN disposition IS NOT NULL AND lower(trim(cast(disposition AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN disposition IS NOT NULL AND lower(trim(cast(disposition AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM med
UNION ALL
SELECT 'state' AS field,
       count(CASE WHEN state IS NOT NULL AND lower(trim(cast(state AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN state IS NOT NULL AND lower(trim(cast(state AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM med
UNION ALL
SELECT 'county' AS field,
       count(CASE WHEN county IS NOT NULL AND lower(trim(cast(county AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN county IS NOT NULL AND lower(trim(cast(county AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM med
UNION ALL
SELECT 'zip' AS field,
       count(CASE WHEN zip IS NOT NULL AND lower(trim(cast(zip AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN zip IS NOT NULL AND lower(trim(cast(zip AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM med
UNION ALL
SELECT 'agency_name' AS field,
       count(CASE WHEN agency_name IS NOT NULL AND lower(trim(cast(agency_name AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) AS populated,
       count(*) AS total,
       round(100.0 * count(CASE WHEN agency_name IS NOT NULL AND lower(trim(cast(agency_name AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM med
ORDER BY pct
""",

"completeness_by_year": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT 'payer' AS field, yr,
       round(100.0 * count(CASE WHEN payer IS NOT NULL AND lower(trim(cast(payer AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM scope GROUP BY yr
UNION ALL
SELECT 'patient_id' AS field, yr,
       round(100.0 * count(CASE WHEN patient_id IS NOT NULL AND lower(trim(cast(patient_id AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM scope GROUP BY yr
UNION ALL
SELECT 'primary_impression' AS field, yr,
       round(100.0 * count(CASE WHEN primary_impression IS NOT NULL AND lower(trim(cast(primary_impression AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM scope GROUP BY yr
UNION ALL
SELECT 'disposition' AS field, yr,
       round(100.0 * count(CASE WHEN disposition IS NOT NULL AND lower(trim(cast(disposition AS string))) NOT IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting') THEN 1 END) / count(*), 1) AS pct
FROM scope GROUP BY yr
ORDER BY field, yr
""",

"payer_values": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT payer, count(*) AS incidents,
       round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
FROM scope GROUP BY payer ORDER BY incidents DESC LIMIT 50
""",

"payer_mix": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT yr, payer_group, count(*) AS incidents FROM scope GROUP BY yr, payer_group ORDER BY yr, incidents DESC
""",

"payer_by_county": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT state, county, payer_group, count(*) AS incidents
FROM scope GROUP BY state, county, payer_group ORDER BY incidents DESC LIMIT 200
""",

"patient_id_check": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT count(*) AS incidents,
       count(DISTINCT patient_id) AS distinct_patients,
       round(count(*) / count(DISTINCT patient_id), 2) AS incidents_per_patient
FROM med
""",

"medicaid_pyramid": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT CASE WHEN encounters = 1 THEN '1'
            WHEN encounters <= 4 THEN '2-4'
            WHEN encounters <= 11 THEN '5-11'
            ELSE '12+' END AS tier,
       count(*) AS patients,
       sum(encounters) AS encounters,
       round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct_patients,
       round(100.0 * sum(encounters) / sum(sum(encounters)) OVER (), 2) AS pct_encounters
FROM (SELECT patient_id, count(*) AS encounters FROM med WHERE patient_id IS NOT NULL GROUP BY patient_id)
GROUP BY 1 ORDER BY 1
""",

"return_intervals": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT count(*) AS repeat_pairs,
       percentile_approx(days_to_next, 0.5) AS median_days,
       round(100.0 * avg(CASE WHEN days_to_next <= 7 THEN 1 ELSE 0 END), 1) AS pct_within_7d,
       round(100.0 * avg(CASE WHEN days_to_next <= 30 THEN 1 ELSE 0 END), 1) AS pct_within_30d
FROM (
  SELECT datediff(lead(incident_date) OVER (PARTITION BY patient_id ORDER BY incident_date),
                  incident_date) AS days_to_next
  FROM med WHERE patient_id IS NOT NULL
)
WHERE days_to_next IS NOT NULL
""",

"top_impressions": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT primary_impression, count(*) AS incidents,
       round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
FROM med GROUP BY primary_impression ORDER BY incidents DESC LIMIT 30
""",

"behavioral_health": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT yr, count(*) AS medicaid_incidents,
       sum(CASE WHEN lower(primary_impression) RLIKE
           'behavioral|psychiatric|anxiety|depress|bipolar|schizo|suicide|substance|alcohol|overdose'
           THEN 1 ELSE 0 END) AS behavioral,
       round(100.0 * avg(CASE WHEN lower(primary_impression) RLIKE
           'behavioral|psychiatric|anxiety|depress|bipolar|schizo|suicide|substance|alcohol|overdose'
           THEN 1 ELSE 0 END), 1) AS pct
FROM med GROUP BY yr ORDER BY yr
""",

"hour_by_day": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT dow, hr, count(*) AS incidents FROM med GROUP BY dow, hr ORDER BY dow, hr
""",

"disposition_mix": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT disposition, count(*) AS incidents FROM med GROUP BY disposition ORDER BY incidents DESC LIMIT 30
""",

"acuity_by_payer": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pat.Patient_ID_Internal                        AS patient_id,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    dis.Disposition_LZ2_Zip_Code                   AS zip,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_elite_dwgmr.fact_incident fi
  LEFT JOIN prod.silver_elite_dwgmr.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_elite_dwgmr.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date)                   AS yr,
    date_format(incident_date, 'yyyy-MM') AS mo,
    hour(incident_date)                   AS hr,
    date_format(incident_date, 'E')       AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting')  THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT acuity, payer_group, count(*) AS incidents FROM scope GROUP BY acuity, payer_group ORDER BY incidents DESC
""",

}

import pandas as pd

results = {}
failed = {}
for name, sql in QUERIES.items():
    try:
        results[name] = spark.sql(sql).toPandas()
        print(name, len(results[name]), "rows")
    except Exception as e:
        failed[name] = str(e)
        print(name, "FAILED:", str(e)[:200])

Every query written out in full. Each one carries its own `WITH flat ... scope ... med` block, so any single string can be copied into the SQL editor and run on its own. That also means the SQL can be validated on the server independently of this notebook.

**The join.** `fact_incident` is the center of a star schema and holds mostly foreign keys, so the readable values come from the dimensions, each joined `Dim_X_FK` to `Dim_X_PK`. All joins are LEFT, so an incident with no payment row still appears with a null payer. A missing dimension row is a finding; an inner join would hide it.

**The steps.** `flat` is the join. `scope` adds date parts and the payer grouping and cuts to 2024 through 2026. `med` is the Medicaid subset. These are CTEs inside each statement, not views, so nothing is created in the catalog.

**The payer rule**, which every Medicaid number depends on, with first match winning:

```sql
CASE
  WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
  WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
  WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
  WHEN payer IS NULL OR lower(trim(payer)) IN (...)              THEN 'Unknown'
  ELSE 'Other/Commercial'
END
```

**Blank** means null or a NEMSIS placeholder such as "Not Recorded" or "Not Applicable". A plain null check would report those fields as complete.

The loop keeps the full error text in `failed`, so a failure tells you what actually went wrong.

To split 911 calls from interfacility transfers, add the dispatch complaint column to the `flat` block and a matching CASE to `scope`. One edit in each query.

In [ ]:
display(results["row_check"])

The fan-out check: incidents against distinct incident ids.

If they differ, a dimension has duplicate keys, the join has multiplied rows, and every count below is inflated. Compare the incidents figure against the plain `fact_incident` count from section 1 as well.

## 4. Volume

In [ ]:
display(results["volume_by_month"])
display(results["volume_by_agency"])
display(results["volume_by_state"])

A month that looks far too small is partial coverage, not a change in demand. Look for a ramp at the start and a drop at the end.

An agency onboarding mid-period looks like growth; one dropping out looks like decline. Neither is about patient demand. The state list is the footprint, which matters because Medicaid programs are state-run.

## 5. Completeness

In [ ]:
display(results["completeness"])
display(results["completeness_by_year"])

How many incidents have a usable value in each field.

This is the direct test of what was described in the meeting: roughly half the patients present, only about a tenth of the information filled in. The by-year view matters more than the average, since a field populated in older years and empty now means a workflow or vendor feed changed.

## 6. Payer mix

In [ ]:
display(results["payer_values"])
display(results["payer_mix"])

Raw values first, then the grouping.

Medicaid is flagged explicitly about 4.4% of the time while a generic "Insurance" value takes 38%, so the Medicaid figure is a floor rather than an estimate until that bucket is understood. Watch Unknown too: if it is large or growing in the recent year, the Medicaid share is understated by an amount we cannot measure.

A rough Genie query put Medicaid near 230,000 patients last year, which is a useful cross-check against 2025 here.

In [ ]:
display(spark.table("prod.silver_elite_dwgmr.dim_payment").limit(20).toPandas())

Twenty rows straight from the payment dimension.

The fastest way at the Insurance bucket question. Look for a secondary method, plan name, or company field carrying the real payer. If Medicaid managed care plan names appear there, widening the Medicaid branch of the CASE is a one line change that moves the share materially.

In [ ]:
display(results["payer_by_county"])

County is the level payer conversations happen at, and the level ACS, HPSA and CDC PLACES join at.

## 7. How often patients come back

In [ ]:
display(results["patient_id_check"])

Read this before anything below it.

If incidents per patient is near 1.0, the identifier is generated per encounter rather than per person. The tiers and intervals below would be measuring nothing, and repeat utilization cannot be built from this table. That is a finding in its own right, since the utilization pyramid and the recurrent demand model are both named as high value products.

In [ ]:
display(results["medicaid_pyramid"])
display(results["return_intervals"])

Medicaid patients grouped into 1, 2-4, 5-11 and 12 or more encounters, then the time to their next encounter.

The point of the pyramid is the gap between percent of patients and percent of encounters: a small group of high utilizers driving a disproportionate share. The 7 and 30 day windows are the two named in the recurrent demand model.

Both are only meaningful if the previous cell showed a patient id that persists across visits.

## 8. Clinical and timing

In [ ]:
display(results["top_impressions"])
display(results["behavioral_health"])

The most common primary impressions among Medicaid incidents, which are the primary ICD-10 codes, then the behavioral health share.

The behavioral figure is the EPCR side counterpart to the nurse navigation note screening. That work reads free text nurse notes; this reads coded impressions. Two independent measurements of the same population are worth more than either alone. This matches full words, avoiding the problem where a short acronym matches inside an unrelated word.

In [ ]:
display(results["hour_by_day"])
display(results["disposition_mix"])
display(results["acuity_by_payer"])

Demand by day and hour, what happened to the patient, and acuity against payer.

The timing claim to test is that weekday afternoons carry the heaviest volume while overnight has the worst response times, which would mean staffing built around a morning peak is aimed at the wrong hours. Days sort alphabetically, so read the labels.

Acuity against payer speaks to whether Medicaid EMS demand is simply low acuity misuse. If the Medicaid distribution resembles the other payer groups, that claim has support in our own data.

## 9. What we can answer

In [ ]:
QUESTIONS = {
    "Medicaid share of EMS volume": ["payer", "incident_date"],
    "Medicaid mix by county": ["county", "payer"],
    "Behavioral health share of Medicaid EMS": ["primary_impression", "payer"],
    "Utilization pyramid (1 / 2-4 / 5-11 / 12+)": ["patient_id", "payer"],
    "7- and 30-day recurrent demand": ["patient_id", "incident_date"],
    "Clinical mix / potentially avoidable episodes": ["primary_impression", "payer"],
    "Time-of-day and day-of-week demand": ["incident_date"],
    "Transport vs non-transport disposition": ["disposition", "payer"],
    "Acuity mix by payer": ["acuity", "payer"],
    "Geographic overlay with ACS / HPSA / PLACES": ["county", "state"],
    "ED outcome linkage": ["incident_id"],
}

pct = results["completeness_medicaid"].set_index("field")["pct"].to_dict()

rows = []
for question, fields in QUESTIONS.items():
    worst = min(pct.get(f, 0.0) for f in fields)
    verdict = "Ready" if worst >= 80 else ("Caveat needed" if worst >= 40 else "Blocked")
    rows.append((question, ", ".join(fields), worst, verdict))

scorecard = pd.DataFrame(rows, columns=["question", "fields", "weakest_pct", "verdict"]) \
              .sort_values("weakest_pct", ascending=False)
results["scorecard"] = scorecard
display(scorecard)

Each question scored by the least complete field it needs. 80 or above is Ready, 40 to 80 needs a caveat, below 40 is blocked.

This is the draft list to take back to Noah and Rex: here is what the data can answer, you tell us which ones matter and what you would do with the answer. Scoring them means the conversation starts from what is possible rather than from a general offer to analyze.

State the limitation whenever this is shown. It measures whether fields are filled in, not whether they are correct or usable. A patient id populated on every row still scores 100 even if it is regenerated each encounter. The outcome linkage row only checks that the EPCR key exists, not that anything matches it.

## 10. Save results

In [ ]:
import os

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
RESULTS = os.path.join("/Workspace", os.path.dirname(nb_path).lstrip("/"), "results")
os.makedirs(RESULTS, exist_ok=True)

path = os.path.join(RESULTS, "ems_data_profiling_results.xlsx")
with pd.ExcelWriter(path, engine="openpyxl") as writer:
    for name, df in results.items():
        df.head(5000).to_excel(writer, sheet_name=name[:31], index=False)

print(path)

One workbook into a `results` folder beside this notebook. A file, not a table.

Everything is already in pandas, so this is a straight write with no further queries. Each sheet caps at 5,000 rows.

This is also the de-identified extract that was asked for: counts and rates only, no patient names, so it can be shared without a data access request.

## 11. Open questions

- What is inside the generic Insurance value on the payer field? Section 6 goes at it. Until that is answered every Medicaid figure here is a floor.
- Do we keep or exclude interfacility transfers? Different answer for a payer view than for the 911 routing question. Needs the dispatch complaint column added to the SQL.
- Does the patient id stay the same across incidents, or get created fresh each call? Section 7 answers it, and all the repeat use work depends on it.
- Does the payer field come from billing, or from the crew at the scene? Revenue cycle data is expected in about three weeks and should settle it.
- Outcomes data: which table, and which column links it to EPCR. Not in this notebook until a readable table is identified.
- Where do the nurse navigation calls and the EPCR records overlap? The link is patients who were transported, since those appear in both.
- Image Trend (`silver_elite_dwamgh`) access, and whether it needs adding for national coverage.